[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/origami/origami.ipynb)

# Folding a Kabuto helmet

Fold a square into a samurai helmet, following the eleven-stage [ganja.js sequence](https://github.com/enkimute/ganja.js/blob/master/examples/example_pga3d_origami.html). Point coincidences and line coincidences construct the creases; motors turn the selected paper faces. Two folds reopen to leave creases for the horns.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D
from examples.animation import save_figure, show_animation
from examples.geometry.origami import render
from examples.geometry.origami.core import Paper

In [ ]:
ga = PGA3D
mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()
Direction = ga.gatype.from_blades("yzw zxw xyw")
Plane = ga.gatype.vector()
Line = ga.gatype.bivector()

In [ ]:
side = 2.0
quarter_turns = np.arange(4) * np.pi / 2
corner: Point = (-mv.x * side + mv.y * side + mv.w).dual()
sheet: Plane = mv.z

# Construct the square in cyclic order, starting at its upper left corner.
corners: Point = (mv.xy * (-quarter_turns / 2)).exp() >> corner
# Tiny display gaps are carried as ideal displacements, independently of the crease geometry.
offsets: Direction = mv("yzw zxw xyw", [[0.0, 0.0, 0.0]])
# A crease plane at any angle to the sheet cuts it in full points, so the paper takes the full point layout.
paper = Paper(corners.cast(Point), np.array([len(corners)]), offsets, sheet)
upper_left, lower_left, lower_right, upper_right = corners
save_figure(render.draw(paper), "origami_square");

In [ ]:
def point_bisector(first: Point, second: Point) -> Plane:
    # A fold taking one point onto the other lies in their perpendicular bisector.
    return ((first & second) | (first + second)).normalized()


def line_bisector(first: Line, second: Line, sheet: Plane) -> Plane:
    first, second = first.normalized(), second.normalized()
    # The plane through the first line perpendicular to the sheet meets the second at their crossing.
    intersection: Point = (sheet | first) ^ second
    return ((first + second) | intersection).normalized()


crease_tolerance = side * 1e-10
first_faces = (0,)
preview_angle = np.pi / 2
crease: Plane = point_bisector(upper_left, lower_right)
hinge: Line = (crease ^ sheet).normalized()

# Splitting inserts (edge_start & edge_end) ^ crease where an edge crosses the plane.
# The returned mask selects the negative-side pieces to turn; the other pieces stay put.
split, turning, _ = paper.split(crease, first_faces, crease_tolerance)
preview = split.folded(hinge, turning, preview_angle)
save_figure(render.draw(preview), "origami_first_fold");

Each selected face splits into its moving piece followed by its stationary piece. The recipe refers to those face indices after each completed stage, letting it fold particular layers. A tiny distance tolerance identifies vertices already on a crease after preceding half-turns.

In [ ]:
centre: Point = corners.mean(axis=0)
top: Point = (upper_right + upper_left) / 2
left: Point = (upper_left + lower_left) / 2
quarter: Point = (upper_left + centre) / 2
upper_centre: Point = (centre + top) / 2
left_centre: Point = (centre + left) / 2
eighth: Point = (centre + quarter) / 2

selections = (
    (0,), (0, 1), (1, 3), (0, 1, 3, 4),  # Triangle, doubled corner flaps, then turn them back.
    (2, 7), (0, 6),                       # Fold and reopen the paired layers of each horn.
    (0, 7), (4, 12),                      # Open the horns, keeping their shared edges joined.
    (8,), (8, 9), (20,),                  # Form the front brim; turn the back sheet underneath.
)
peaks = np.pi * np.array([0.5, 0.5, 0.5, 0.5, 1, 1, 0.5, 0.5, 0.5, 0.5, -0.5])
ends = np.pi * np.array([1, 1, 1, 1, 0, 0, 1, 1, 1, 1, -1])

# The same three constructions specify all eleven creases.
planes: Plane = stack([
    point_bisector(upper_left, lower_right),
    point_bisector(upper_right, lower_right),
    point_bisector(lower_left, lower_right),
    point_bisector(centre, upper_left),
    point_bisector(top, centre),
    point_bisector(left, centre),
    line_bisector(quarter & left_centre, centre & quarter, sheet),
    line_bisector(upper_centre & quarter, quarter & centre, sheet),
    point_bisector(eighth, upper_left),
    (sheet | (top & left)).normalized(),
    (sheet | (top & left)).normalized(),
])

In [ ]:
half_steps = 8
duration_ms = 65
fractions = np.linspace(0, 1, half_steps + 1)[1:]
fractions = fractions * fractions * (3 - 2 * fractions)


def folding() -> Iterator[Paper]:
    current = paper
    yield current
    for plane, selected, peak, end in zip(planes, selections, peaks, ends):
        # Split once per crease; all animation frames use exactly the same pieces.
        split, turning, _ = current.split(plane, selected, crease_tolerance)
        hinge: Line = plane ^ sheet
        for angle in np.concatenate([peak * fractions, peak + (end - peak) * fractions]):
            current = split.folded(hinge, turning, angle)
            yield current


frames = render.animate(folding())
show_animation(frames, "origami_kabuto", duration_ms)

Each doubled flap moves together along its shared edge; the final flap folds behind the helmet.